# 🎤 Parkinson's Disease Detection from Speech Audio

## Multilingual XLS-R Transfer Learning + ML Ensemble

**🎯 Approach: Pre-trained Multilingual Speech Model + Classical ML + Stacking Ensemble**

### Key Design:
- ✅ **XLS-R 300M (Facebook/Meta)** — Pre-trained on **436K hours of speech in 128 languages**
  - Frozen feature extractor → 1024-dim embeddings per audio
  - Language-agnostic: works on English, Italian, Telugu, Hindi, and 124+ more
  - Captures deep speech representations that hand-crafted features miss
- ✅ **Hand-crafted voice features** — MFCC + Delta + Jitter/Shimmer/HNR + Spectral
- ✅ **Italian Parkinson's Voice & Speech Dataset** — 28 PD + 37 HC patients (831 recordings)
- ✅ **4-Path DL Fusion** — XLS-R MLP + CNN (mel-spec) + MFCC MLP + Acoustic MLP with SE Attention
- ✅ **ML Ensemble** — RF + SVM + XGBoost + GradientBoosting + LightGBM (5 models)
- ✅ **SMOTE oversampling** — Balances minority class for ML training
- ✅ **Stacking Meta-Learner** — Combines all 6 model predictions optimally
- ✅ **Patient-Level Stratified Group 5-Fold CV** — No data leakage
### Why XLS-R (Multilingual Wav2Vec2)?
- Pre-trained on **436,000+ hours** of speech in **128 languages** (VoxPopuli, CommonVoice, BABEL, MLS)
- Language-agnostic: PD voice biomarkers (dysarthria, tremor, breathiness) transfer across languages
- 1024-dim embeddings — richer representations than base wav2vec2 (768-dim)
- Published studies show **85–95% accuracy** on PD speech detection
- Works on **any language** without retraining — train on Italian, deploy on English/Telugu/Hindi

### 📁 Dataset: Italian Parkinson's Voice and Speech (IEEE DataPort)
- **28 PD patients** (437 recordings) — sustained vowels /a/,/e/,/i/,/o/,/u/, breathing, DDK
- **22 Elderly Healthy Controls** (349 recordings)
- **15 Young Healthy Controls** (45 recordings)
- Clinical microphone, controlled environment, 16-bit PCM

---
**Works on Kaggle, Google Colab, and Local!**

- Enable GPU: Settings → Accelerator → GPU T4 x2 (recommended)- Enable GPU: Settings → Accelerator → GPU T4 x2 (recommended)

In [ ]:
# ============================================
# 📦 Install Dependencies
# ============================================
# NOTE: Do NOT reinstall torch/torchaudio/torchvision on Kaggle/Colab —
# they come pre-installed with CUDA support. Reinstalling via pip
# overwrites them with the CPU-only build and disables GPU!

import os
_IN_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
_IN_COLAB = 'google.colab' in str(globals().get('get_ipython', lambda: None)())

if not (_IN_KAGGLE or _IN_COLAB):
    # Local: install PyTorch (user should use the correct CUDA version from pytorch.org)
    !pip install -q torch torchaudio torchvision

!pip install -q librosa soundfile
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q praat-parselmouth  # For pitch/formant analysis
!pip install -q pydub  # For audio format conversion
!pip install -q tqdm
!pip install -q transformers  # For XLS-R multilingual pre-trained speech model
!pip install -q xgboost
!pip install -q imbalanced-learn lightgbm

import torch
print("✅ All packages installed successfully!")
print(f"🔥 PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================
# 🖥️ Environment Detection (Kaggle / Colab / Local)
# ============================================

import os, torch

IN_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IN_COLAB = 'google.colab' in str(globals().get('get_ipython', lambda: None)())

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f"🖥️ Environment: {'Kaggle' if IN_KAGGLE else ('Colab' if IN_COLAB else 'Local')}")
print(f"🔧 Device: {DEVICE} ({GPU_NAME})")
print(f"🔥 CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"💾 GPU Memory: {vram:.1f} GB")

In [ ]:
# ============================================
# CELL 3: Dataset Paths Setup
# ============================================

import os
import subprocess
from pathlib import Path

GITHUB_REPO = "https://github.com/Tvenkatathanuj/SDP.git"
REPO_NAME = "SDP"

# Clone repository if running on Kaggle or Colab
if IN_COLAB or IN_KAGGLE:
    if not Path(REPO_NAME).exists():
        print(f"📥 Cloning repository from {GITHUB_REPO}...")
        print("   ⏳ This includes ~810 MB of audio data — may take a few minutes...")
        result = subprocess.run(
            ['git', 'clone', '--depth', '1', GITHUB_REPO],
            capture_output=True, text=True
        )
        if result.returncode == 0 or Path(REPO_NAME).exists():
            print("✅ Repository cloned successfully!")
        else:
            print(f"❌ Clone failed: {result.stderr}")
    else:
        print("✅ Repository already exists")
    DATASET_ROOT = Path(REPO_NAME)
else:
    DATASET_ROOT = Path('.')

# ========== PRIMARY: Italian Parkinson's Voice and Speech Dataset ==========
ITALIAN_PD_DIR = DATASET_ROOT / "Italian Parkinson's Voice and speech"

# ========== SECONDARY (optional): Previous denoised/original datasets ==========
SPEECH_DENOISED_DIR = DATASET_ROOT / "denoised-speech-dataset"
SPEECH_ORIGINAL_DIR = DATASET_ROOT / "original-speech-dataset"
if not SPEECH_DENOISED_DIR.exists():
    SPEECH_DENOISED_DIR = DATASET_ROOT / "speech" / "Parkinson-Patient-Speech-Dataset" / "denoised-speech-dataset"
    SPEECH_ORIGINAL_DIR = DATASET_ROOT / "speech" / "Parkinson-Patient-Speech-Dataset" / "original-speech-dataset"

print(f"\n📁 Dataset Root: {DATASET_ROOT.absolute()}")
print(f"   🇮🇹 Italian PD Dataset: {'✅ Found' if ITALIAN_PD_DIR.exists() else '❌ Missing'} ({ITALIAN_PD_DIR})")
print(f"   Denoised Speech:       {'✅ Found' if SPEECH_DENOISED_DIR.exists() else '⬜ Not found (optional)'}")
print(f"   Original Speech:       {'✅ Found' if SPEECH_ORIGINAL_DIR.exists() else '⬜ Not found (optional)'}")

AUDIO_DIRS = []
if ITALIAN_PD_DIR.exists():
    AUDIO_DIRS.append(('italian_pd', ITALIAN_PD_DIR))
if SPEECH_DENOISED_DIR.exists():
    AUDIO_DIRS.append(('denoised', SPEECH_DENOISED_DIR))
if SPEECH_ORIGINAL_DIR.exists():
    AUDIO_DIRS.append(('original', SPEECH_ORIGINAL_DIR))

print(f"\n   🎯 Using {len(AUDIO_DIRS)} dataset(s): {[d[0] for d in AUDIO_DIRS]}")
if not AUDIO_DIRS:
    print("\n⚠️ No audio datasets found! Place the Italian PD dataset folder in the project root.")

In [ ]:
# ============================================
# 📚 Import Libraries
# ============================================

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW

import torchaudio
import torchaudio.transforms as T
import librosa
import soundfile as sf

# XLS-R / Wav2Vec2 — multilingual pre-trained speech model
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor as Wav2Vec2Processor

try:
    import parselmouth
    from parselmouth.praat import call
    PARSELMOUTH_AVAILABLE = True
except ImportError:
    PARSELMOUTH_AVAILABLE = False
    print("⚠️ Parselmouth not available. Installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'praat-parselmouth'])
    try:
        import parselmouth
        from parselmouth.praat import call
        PARSELMOUTH_AVAILABLE = True
    except:
        PARSELMOUTH_AVAILABLE = False

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    print("⚠️ XGBoost not available. Installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'xgboost'])
    import xgboost as xgb
    XGB_AVAILABLE = True

# LightGBM — often outperforms XGBoost on small datasets
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False

# SMOTE oversampling — critical for extreme class imbalance
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    balanced_accuracy_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

print(f"   Parselmouth (pitch/formant): {'✅ Available' if PARSELMOUTH_AVAILABLE else '❌ Not available'}")
print(f"   XGBoost: {'✅ Available' if XGB_AVAILABLE else '❌ Not available'}")
print(f"   LightGBM: {'✅ Available' if LGB_AVAILABLE else '❌ Not available'}")
print(f"   SMOTE oversampling: {'✅ Available' if SMOTE_AVAILABLE else '❌ Not available'}")
print(f"   XLS-R (HuggingFace): ✅ Available")
print("✅ All libraries imported successfully!")

In [ ]:
# ============================================
# ⚙️ Configuration — XLS-R Multilingual + Enhanced ML Ensemble
# ============================================

@dataclass
class AudioConfig:
    """Configuration for voice-based Parkinson's detection with XLS-R multilingual model."""
    
    # Paths
    checkpoint_dir: str = "./checkpoints_audio"
    
    # Audio Processing
    sample_rate: int = 16000
    max_audio_length: int = 8  # seconds (longer clips capture more speech patterns)
    n_mfcc: int = 40
    n_mels: int = 128
    n_fft: int = 2048
    hop_length: int = 512
    
    # Enhanced features
    use_delta_mfcc: bool = True
    use_spectral_contrast: bool = True
    use_chroma: bool = True
    use_tonnetz: bool = True
    
    # === XLS-R 300M (Multilingual Wav2Vec2) ===
    # 128 languages, 300M params, 1024-dim output — best for cross-lingual PD detection
    wav2vec2_model_name: str = "facebook/wav2vec2-xls-r-300m"
    wav2vec2_embed_dim: int = 1024  # Output embedding dimension (1024 for XLS-R 300M)
    freeze_wav2vec2: bool = True    # Freeze backbone (use as feature extractor only)
    
    # Model Architecture
    hidden_dim: int = 256
    dropout: float = 0.4
    
    # Training
    batch_size: int = 16
    num_epochs: int = 100
    learning_rate: float = 1e-4
    weight_decay: float = 0.03
    n_folds: int = 5
    
    # Loss & Optimization
    focal_alpha: float = 0.75  # ML uses this directly; DL overrides to 0.5 since WeightedRandomSampler balances batches
    focal_gamma: float = 3.0
    label_smoothing: float = 0.05
    optimize_threshold: bool = True
    patience: int = 20  # Early stopping patience (epochs without improvement)
    
    # Augmentation  
    use_augmentation: bool = True
    time_stretch_rate: Tuple[float, float] = (0.85, 1.15)
    pitch_shift_steps: int = 3
    noise_factor: float = 0.008
    spec_augment: bool = True
    freq_mask_param: int = 15
    time_mask_param: int = 25
    
    # Mixup augmentation (proven to boost generalization on small datasets)
    use_mixup: bool = True
    mixup_alpha: float = 0.4
    
    # Misc
    seed: int = 42
    num_workers: int = 0

config = AudioConfig()
os.makedirs(config.checkpoint_dir, exist_ok=True)

# Calculate expected feature dimensions
n_acoustic_base = 14  # jitter, shimmer, HNR, formants, spectral, etc.
n_mfcc_features = config.n_mfcc * 2  # mean + std
n_delta = config.n_mfcc * 2 if config.use_delta_mfcc else 0
n_contrast = 7 if config.use_spectral_contrast else 0
n_chroma = 12 if config.use_chroma else 0
n_tonnetz = 6 if config.use_tonnetz else 0
total_handcrafted = n_mfcc_features + n_delta + n_acoustic_base + n_contrast + n_chroma + n_tonnetz
total_w2v = config.wav2vec2_embed_dim

print("✅ Audio Configuration loaded!")
print(f"   Sample Rate: {config.sample_rate} Hz | Max Length: {config.max_audio_length}s")
print(f"   Batch Size: {config.batch_size} | Epochs: {config.num_epochs}")
print(f"   Dropout: {config.dropout} | Weight Decay: {config.weight_decay}")

print(f"\n   🧠 XLS-R: {config.wav2vec2_model_name}")
print(f"   XLS-R Embedding: {config.wav2vec2_embed_dim} dims (frozen={config.freeze_wav2vec2}, 128 languages)")
print(f"   📊 Hand-crafted features: {total_handcrafted} dims")
print(f"   📊 Total ML features per sample: {total_handcrafted + total_w2v} dims")

In [ ]:
# ============================================
# 🎲 Set Random Seeds for Reproducibility
# ============================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)
print(f"✅ Random seed set to {config.seed}")

In [ ]:
# ============================================
# 🎤 Feature Extractor — XLS-R (Multilingual) + Hand-Crafted Features
# ============================================

class Wav2Vec2Extractor:
    """Extract XLS-R multilingual embeddings from raw audio (frozen, no gradient).
    
    Pre-trained on 436K hours of speech in 128 languages — captures
    language-agnostic speech patterns including dysarthria, breathiness, and tremor.
    """
    
    def __init__(self, config: AudioConfig, device=None):
        self.config = config
        self.device = device or DEVICE
        
        print(f"   Loading XLS-R: {config.wav2vec2_model_name}...")
        self.processor = Wav2Vec2Processor.from_pretrained(config.wav2vec2_model_name)
        self.model = Wav2Vec2Model.from_pretrained(config.wav2vec2_model_name)
        self.model.eval()
        self.model.to(self.device)
        
        # Freeze all parameters
        if config.freeze_wav2vec2:
            for param in self.model.parameters():
                param.requires_grad = False
        
        print(f"   ✅ XLS-R loaded ({sum(p.numel() for p in self.model.parameters())/1e6:.1f}M params, frozen={config.freeze_wav2vec2})")
    
    @torch.no_grad()
    def extract_embedding(self, waveform: np.ndarray) -> np.ndarray:
        """Extract embedding from a single waveform.
        
        Returns mean-pooled hidden states (config.wav2vec2_embed_dim dims).
        """
        # Processor expects float32, 16kHz
        inputs = self.processor(
            waveform, 
            sampling_rate=self.config.sample_rate, 
            return_tensors="pt",
            padding=True
        )
        input_values = inputs.input_values.to(self.device)
        
        outputs = self.model(input_values)
        # Mean-pool over time dimension → (embed_dim,)
        hidden_states = outputs.last_hidden_state  # (1, T, embed_dim)
        embedding = hidden_states.mean(dim=1).squeeze(0).cpu().numpy()  # (embed_dim,)
        
        return embedding
    
    def extract_batch_embeddings(self, waveforms: List[np.ndarray]) -> np.ndarray:
        """Extract embeddings for a batch of waveforms."""
        embeddings = []
        for wf in waveforms:
            emb = self.extract_embedding(wf)
            embeddings.append(emb)
        return np.array(embeddings)


class AudioFeatureExtractor:
    """Extract comprehensive voice features for Parkinson's detection.
    
    Combines:
    1. XLS-R embeddings (1024 dims) — multilingual deep speech representations
    2. MFCC + Delta (160 dims) — spectral envelope
    3. Voice quality (14 dims) — jitter, shimmer, HNR, formants 
    4. Spectral extras (25 dims) — contrast, chroma, tonnetz
    """
    
    def __init__(self, config: AudioConfig):
        self.config = config
        self.sr = config.sample_rate
        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=self.sr, n_fft=config.n_fft,
            hop_length=config.hop_length, n_mels=config.n_mels, power=2.0
        )
        
    def extract_mfcc(self, waveform: np.ndarray) -> np.ndarray:
        """Extract MFCC + optional delta/delta-delta."""
        mfcc = librosa.feature.mfcc(
            y=waveform, sr=self.sr, n_mfcc=self.config.n_mfcc,
            n_fft=self.config.n_fft, hop_length=self.config.hop_length
        )
        features = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
        
        if self.config.use_delta_mfcc:
            delta_mfcc = librosa.feature.delta(mfcc, order=1)
            features = np.concatenate([
                features,
                np.mean(delta_mfcc, axis=1),
                np.std(delta_mfcc, axis=1)
            ])
        
        return features
    
    def extract_voice_quality_features(self, waveform: np.ndarray) -> Dict[str, float]:
        """Extract clinical voice quality markers."""
        features = {}
        
        if PARSELMOUTH_AVAILABLE and len(waveform) > self.sr * 0.5:
            try:
                sound = parselmouth.Sound(waveform, sampling_frequency=self.sr)
                pitch = sound.to_pitch(time_step=0.01)
                features['mean_pitch'] = call(pitch, "Get mean", 0, 0, "Hertz")
                features['std_pitch'] = call(pitch, "Get standard deviation", 0, 0, "Hertz")
                pp = call(sound, "To PointProcess (periodic, cc)", 75, 500)
                features['jitter_local'] = call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
                features['jitter_rap'] = call(pp, "Get jitter (rap)", 0, 0, 0.0001, 0.02, 1.3)
                features['shimmer_local'] = call([sound, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
                harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
                features['hnr'] = call(harmonicity, "Get mean", 0, 0)
                formants = sound.to_formant_burg(time_step=0.01)
                features['f1_mean'] = call(formants, "Get mean", 1, 0, 0, "Hertz")
                features['f2_mean'] = call(formants, "Get mean", 2, 0, 0, "Hertz")
                features['f3_mean'] = call(formants, "Get mean", 3, 0, 0, "Hertz")
            except Exception:
                for key in ['mean_pitch', 'std_pitch', 'jitter_local', 'jitter_rap', 
                           'shimmer_local', 'hnr', 'f1_mean', 'f2_mean', 'f3_mean']:
                    features[key] = 0.0
        else:
            for key in ['mean_pitch', 'std_pitch', 'jitter_local', 'jitter_rap', 
                       'shimmer_local', 'hnr', 'f1_mean', 'f2_mean', 'f3_mean']:
                features[key] = 0.0
        
        features['spectral_centroid'] = np.mean(librosa.feature.spectral_centroid(y=waveform, sr=self.sr))
        features['spectral_rolloff'] = np.mean(librosa.feature.spectral_rolloff(y=waveform, sr=self.sr))
        features['zcr'] = np.mean(librosa.feature.zero_crossing_rate(waveform))
        rms = librosa.feature.rms(y=waveform)
        features['rms_mean'] = np.mean(rms)
        features['rms_std'] = np.std(rms)
        
        return features
    
    def extract_extra_features(self, waveform: np.ndarray) -> np.ndarray:
        """Extract spectral contrast, chroma, tonnetz."""
        extras = []
        if self.config.use_spectral_contrast:
            sc = librosa.feature.spectral_contrast(y=waveform, sr=self.sr, n_bands=6)
            extras.append(np.mean(sc, axis=1))
        if self.config.use_chroma:
            chroma = librosa.feature.chroma_stft(y=waveform, sr=self.sr)
            extras.append(np.mean(chroma, axis=1))
        if self.config.use_tonnetz:
            try:
                tonnetz = librosa.feature.tonnetz(y=waveform, sr=self.sr)
                extras.append(np.mean(tonnetz, axis=1))
            except:
                extras.append(np.zeros(6))
        if extras:
            return np.concatenate(extras)
        return np.array([])
    
    def extract_mel_spectrogram(self, waveform: torch.Tensor) -> torch.Tensor:
        """Extract mel-spectrogram for CNN."""
        mel_spec = self.mel_spectrogram(waveform)
        mel_spec_db = T.AmplitudeToDB()(mel_spec)
        return mel_spec_db
    
    def __call__(self, waveform: np.ndarray) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Extract all hand-crafted features from voice audio.
        
        Returns:
            mel_spec: Mel-spectrogram (C, H, W) for CNN
            mfcc_features: MFCC + delta features for MLP
            acoustic_features: Voice quality + extra features for MLP/ML
        """
        waveform_tensor = torch.from_numpy(waveform).float().unsqueeze(0)
        mel_spec = self.extract_mel_spectrogram(waveform_tensor)
        
        mfcc = self.extract_mfcc(waveform)
        mfcc_tensor = torch.from_numpy(mfcc).float()
        
        voice_features = self.extract_voice_quality_features(waveform)
        voice_vals = list(voice_features.values())
        extra = self.extract_extra_features(waveform)
        all_acoustic = np.concatenate([voice_vals, extra]) if len(extra) > 0 else np.array(voice_vals)
        acoustic_tensor = torch.tensor(all_acoustic, dtype=torch.float32)
        
        return mel_spec, mfcc_tensor, acoustic_tensor

print("✅ Feature extractors defined!")
print(f"   🧠 Wav2Vec2Extractor: {config.wav2vec2_embed_dim}-dim deep speech embeddings (multilingual, 128 languages)")
print("   🎵 AudioFeatureExtractor: MFCC + Voice Quality + Spectral")

In [ ]:
# ============================================
# CELL 8: Load ALL Audio Datasets
# ============================================

def load_italian_pd_dataset(dataset_dir: Path) -> pd.DataFrame:
    """Load the Italian Parkinson's Voice and Speech dataset.
    
    Structure:
      28 People with Parkinson's disease/  →  label=1
        1-5/, 6-10/, 11-16/, 17-28/  →  patient subfolders
      22 Elderly Healthy Control/          →  label=0
        AGNESE P/, ANGELA C/, ...          →  patient subfolders
      15 Young Healthy Control/            →  label=0
        Alberto R/, Alessandro M/, ...     →  patient subfolders
    """
    data = []
    
    group_configs = [
        ("28 People with Parkinson's disease", 1, "PD"),
        ("22 Elderly Healthy Control", 0, "EHC"),
        ("15 Young Healthy Control", 0, "YHC"),
    ]
    
    for group_folder, label, group_tag in group_configs:
        group_path = dataset_dir / group_folder
        if not group_path.exists():
            print(f"   ⚠️ Missing: {group_path}")
            continue
        
        # Find all wav files recursively
        wav_files = list(group_path.rglob('*.wav'))
        
        for wav_file in wav_files:
            try:
                rel_path = wav_file.relative_to(group_path)
            except ValueError:
                continue
            
            # Determine patient_id from folder structure
            parts = rel_path.parts
            if label == 1:
                # PD: group_folder / range_folder / patient_name / file.wav
                if len(parts) >= 2:
                    patient_name = parts[-2]  # Parent folder = patient name
                else:
                    patient_name = parts[0]
                patient_id = f"PD_{patient_name.replace(' ', '_')}"
            else:
                # HC: group_folder / patient_name / file.wav
                if len(parts) >= 2:
                    patient_name = parts[0]  # First subfolder = patient name
                else:
                    patient_name = wav_file.stem[:10]
                patient_id = f"{group_tag}_{patient_name.replace(' ', '_')}"
            
            try:
                info = sf.info(str(wav_file))
                duration = info.duration
                sr = info.samplerate
            except:
                continue
            
            if duration < 0.3:
                continue
            
            data.append({
                'patient_id': patient_id,
                'file_path': str(wav_file),
                'file_name': wav_file.name,
                'duration': duration,
                'sample_rate': sr,
                'label': label,
                'source': 'italian_pd',
                'group': group_tag,
                'description': "Parkinson's" if label == 1 else "Healthy Control"
            })
    
    return pd.DataFrame(data)


def load_legacy_dataset(audio_dir: Path, source_tag: str) -> pd.DataFrame:
    """Load audio files from the old denoised/original directories."""
    data = []
    pd_folders = ['DL', 'LW', 'Tessi']
    hc_folders = ['emma', 'Faces']
    patient_labels = {
        'DL': 1, 'LW': 1, 'Tessi': 1, 'ES': 1, 'SI': 1,
        'emma': 0, 'Faces': 0, 'IC': 0, 'WP': 0,
        'BG': 0, 'JC': 0, 'MJ': 0, 'SK': 0, 'TP': 0, 'TS': 0,
    }
    
    audio_files = list(audio_dir.rglob('*.wav'))
    
    for audio_file in audio_files:
        try:
            rel_path = audio_file.relative_to(audio_dir)
        except ValueError:
            continue
        path_parts = rel_path.parts
        if len(path_parts) < 2:
            continue
        
        main_folder = path_parts[0]
        
        if main_folder in pd_folders:
            label = 1
            patient_id = f"OLD_{main_folder}"
        elif main_folder in hc_folders:
            label = 0
            if len(path_parts) > 2:
                subfolder = path_parts[1]
                clean = subfolder.replace('_au', '').replace('_ori', '').replace('1111', '')
                patient_id = f"OLD_{clean}"
            else:
                patient_id = f"OLD_{main_folder}"
        else:
            file_prefix = audio_file.stem[:2]
            if file_prefix in patient_labels:
                label = patient_labels[file_prefix]
                patient_id = f"OLD_{file_prefix}"
            else:
                continue
        
        try:
            info = sf.info(str(audio_file))
            duration = info.duration
            sr = info.samplerate
        except:
            continue
        
        if duration < 0.3:
            continue
        
        data.append({
            'patient_id': patient_id,
            'file_path': str(audio_file),
            'file_name': audio_file.name,
            'duration': duration,
            'sample_rate': sr,
            'label': label,
            'source': source_tag,
            'group': 'legacy',
            'description': "Parkinson's" if label == 1 else "Healthy Control"
        })
    
    return pd.DataFrame(data)


# ============ Load ALL datasets ============
print("📊 Loading audio datasets...")
all_dfs = []

for tag, dir_path in AUDIO_DIRS:
    print(f"\n📂 Scanning {tag}: {dir_path}")
    if tag == 'italian_pd':
        df_part = load_italian_pd_dataset(dir_path)
    else:
        df_part = load_legacy_dataset(dir_path, tag)
    
    if len(df_part) > 0:
        print(f"   Found {len(df_part)} voice samples from {df_part['patient_id'].nunique()} patients")
        all_dfs.append(df_part)

if all_dfs:
    df_audio = pd.concat(all_dfs, ignore_index=True)
    
    print(f"\n" + "="*60)
    print(f"✅ Combined Audio Dataset loaded!")
    print(f"="*60)
    print(f"   Total voice samples: {len(df_audio)}")
    print(f"   Unique patients: {df_audio['patient_id'].nunique()}")
    print(f"   Sources: {df_audio['source'].value_counts().to_dict()}")
    
    print(f"\n📈 Class Distribution:")
    class_counts = df_audio['label'].value_counts()
    n_pd = class_counts.get(1, 0)
    n_hc = class_counts.get(0, 0)
    print(f"   Healthy (0):     {n_hc} samples")
    print(f"   Parkinson's (1): {n_pd} samples")
    ratio = n_hc / max(n_pd, 1)
    print(f"   Ratio HC:PD = {ratio:.1f}:1  {'✅ Good balance' if ratio < 3 else '⚠️ Imbalanced'}")
    
    print(f"\n👥 Samples per Patient (top 20):")
    patient_summary = df_audio.groupby('patient_id').agg({
        'label': 'first',
        'file_name': 'count',
        'duration': 'mean'
    }).rename(columns={'file_name': 'n_files', 'duration': 'avg_duration'})
    patient_summary = patient_summary.sort_values(['label', 'n_files'], ascending=[False, False])
    print(patient_summary.head(20))
    
    print(f"\n   PD patients: {df_audio[df_audio['label']==1]['patient_id'].nunique()}")
    print(f"   HC patients: {df_audio[df_audio['label']==0]['patient_id'].nunique()}")
    
    print(f"\n⏱️ Audio Statistics:")
    print(f"   Average duration: {df_audio['duration'].mean():.2f}s")
    print(f"   Min: {df_audio['duration'].min():.2f}s | Max: {df_audio['duration'].max():.2f}s")
else:
    df_audio = pd.DataFrame()
    print("\n❌ No audio files loaded!")

---
### ⚠️ **Dataset: Italian Parkinson's Voice and Speech (IEEE DataPort)**

The notebook automatically loads the Italian PD dataset from the GitHub repository.

**Expected structure:**
```
Italian Parkinson's Voice and speech/
  28 People with Parkinson's disease/
    1-5/
      Roberto R/
        recording1.wav
    6-10/
      ...
    11-16/
      ...
    17-28/
      ...
  22 Elderly Healthy Control/
    AGNESE P/
      recording1.wav
    ...
  15 Young Healthy Control/
    Alberto R/
      recording1.wav
    ...
```

**On Colab/Kaggle:** The dataset is cloned automatically from GitHub (~810 MB).
**Locally:** Place the dataset folder in the project root (same level as this notebook).

---

In [ ]:
# ============================================
# CELL 9: Patient-Level Stratified Group K-Fold CV Setup
# ============================================
# CRITICAL: Folds are auto-limited to the number of PD patient groups
# so that EVERY test fold contains at least 1 PD patient.

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

def make_weighted_sampler(labels):
    """WeightedRandomSampler to oversample minority class."""
    class_counts = np.bincount(labels, minlength=2)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).float(),
        num_samples=len(labels), replacement=True
    )

if len(df_audio) > 0:
    groups = df_audio['patient_id'].values
    labels_arr = df_audio['label'].values
    n_unique_groups = len(np.unique(groups))

    # Count PD patient groups specifically — this limits max useful folds
    pd_mask = labels_arr == 1
    pd_groups = np.unique(groups[pd_mask])
    hc_groups = np.unique(groups[~pd_mask])
    n_pd_groups = len(pd_groups)
    n_hc_groups = len(hc_groups)

    print(f"\n📊 Patient Group Analysis:")
    print(f"   PD  patient groups ({n_pd_groups}): {sorted(pd_groups)}")
    print(f"   HC  patient groups ({n_hc_groups}): {sorted(hc_groups)}")
    print(f"   Total: {n_unique_groups} unique patient groups")

    # Limit folds to n_pd_groups so every fold gets PD in test
    actual_folds = min(config.n_folds, n_pd_groups, n_hc_groups)
    if actual_folds < config.n_folds:
        print(f"\n   ⚠️  Reducing from {config.n_folds} to {actual_folds} folds")
        print(f"       (only {n_pd_groups} PD groups — need ≥1 PD per test fold)")

    if actual_folds < 2:
        print(f"\n   ⚠️  Only {n_pd_groups} PD group(s) — using holdout split instead of CV")
        # Fallback: single train/val/test split
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=config.seed)
        train_val_idx, test_idx = next(gss.split(df_audio, df_audio['label'], groups))
        test_df = df_audio.iloc[test_idx].reset_index(drop=True)
        train_val_df = df_audio.iloc[train_val_idx].reset_index(drop=True)
        train_df, val_df = train_test_split(
            train_val_df, test_size=0.15, stratify=train_val_df['label'],
            random_state=config.seed
        )
        cv_splits = [{
            'fold_name': 'Holdout',
            'train_df': train_df.reset_index(drop=True),
            'val_df': val_df.reset_index(drop=True),
            'test_df': test_df,
        }]
    else:
        sgkf = StratifiedGroupKFold(n_splits=actual_folds, shuffle=True, random_state=config.seed)
        cv_splits = []

        for fold_idx, (train_val_idx, test_idx) in enumerate(sgkf.split(df_audio, df_audio['label'], groups)):
            test_df = df_audio.iloc[test_idx].reset_index(drop=True)
            train_val_df = df_audio.iloc[train_val_idx].reset_index(drop=True)

            # Patient-level train/val split — no patient overlap
            tv_groups = train_val_df['patient_id'].values
            n_tv_groups = len(np.unique(tv_groups))

            if n_tv_groups >= 3:
                gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=config.seed)
                train_inner, val_inner = next(gss.split(train_val_df, train_val_df['label'], tv_groups))
                train_df = train_val_df.iloc[train_inner].reset_index(drop=True)
                val_df = train_val_df.iloc[val_inner].reset_index(drop=True)
            else:
                train_df, val_df = train_test_split(
                    train_val_df, test_size=0.15, stratify=train_val_df['label'],
                    random_state=config.seed
                )

            cv_splits.append({
                'fold_name': f'Fold-{fold_idx+1}',
                'train_df': train_df.reset_index(drop=True),
                'val_df': val_df.reset_index(drop=True),
                'test_df': test_df,
            })

    print("\n" + "=" * 70)
    print(f"📊 Patient-Level {len(cv_splits)}-Fold Group CV")
    print(f"   ⚠️  NO patient overlap between train/val/test — prevents data leakage!")
    print(f"   Total: {len(df_audio)} voice samples from {n_unique_groups} unique patients")
    print("=" * 70)
    all_folds_valid = True
    for i, s in enumerate(cv_splits):
        tr_pd = (s['train_df']['label'] == 1).sum()
        tr_hc = (s['train_df']['label'] == 0).sum()
        te_pd = (s['test_df']['label'] == 1).sum()
        te_hc = (s['test_df']['label'] == 0).sum()
        vl_pd = (s['val_df']['label'] == 1).sum()
        tr_p = s['train_df']['patient_id'].nunique()
        te_p = s['test_df']['patient_id'].nunique()
        vl_p = s['val_df']['patient_id'].nunique()

        train_pids = set(s['train_df']['patient_id'].unique())
        test_pids = set(s['test_df']['patient_id'].unique())
        val_pids = set(s['val_df']['patient_id'].unique())
        leak = (train_pids & test_pids) | (val_pids & test_pids)

        if te_pd == 0:
            status = '❌ NO PD IN TEST — fold will be skipped'
            all_folds_valid = False
        elif leak:
            status = f'⚠️ LEAK: {leak}'
        else:
            status = '✅ OK'

        print(f"   {s['fold_name']}: Train={len(s['train_df']):4d} ({tr_p}p PD:{tr_pd:3d} HC:{tr_hc:3d}) | "
              f"Val={len(s['val_df']):3d} ({vl_p}p PD:{vl_pd}) | "
              f"Test={len(s['test_df']):3d} ({te_p}p PD:{te_pd:2d} HC:{te_hc:3d}) {status}")

    if not all_folds_valid:
        print(f"\n   ⚠️  Some folds have no PD in test — they will be SKIPPED during training.")
    print("=" * 70)
else:
    print("⚠️ No data to create CV splits.")

In [ ]:
# ============================================
# 🧠 Pre-Compute ALL Features (XLS-R + Mel + MFCC + Acoustic)
# ============================================
# Pre-computing EVERYTHING eliminates the CPU bottleneck during training.
# Without this, librosa.load + MFCC + parselmouth run in __getitem__
# every epoch, starving the GPU (0% utilization, ~800s/epoch).
# With caching: GPU stays busy, ~10-30s/epoch on P100.

feature_extractor = AudioFeatureExtractor(config)

if len(df_audio) > 0:
    print("🧠 Pre-computing ALL features (XLS-R + mel + MFCC + acoustic)...")
    print(f"   Model: {config.wav2vec2_model_name} (128 languages)")
    print(f"   Files: {len(df_audio)}")

    w2v_extractor = Wav2Vec2Extractor(config, device=DEVICE)

    feature_cache = {}
    max_samples = config.sample_rate * config.max_audio_length

    for idx, row in tqdm(df_audio.iterrows(), total=len(df_audio), desc="   Extracting features"):
        fp = row['file_path']
        try:
            waveform, _ = librosa.load(fp, sr=config.sample_rate, mono=True)
            if np.max(np.abs(waveform)) > 0:
                waveform = waveform / np.max(np.abs(waveform))
            if len(waveform) < max_samples:
                waveform = np.pad(waveform, (0, max_samples - len(waveform)))
            else:
                waveform = waveform[:max_samples]

            # XLS-R embedding (1024-dim)
            w2v_emb = w2v_extractor.extract_embedding(waveform)

            # Hand-crafted features (mel-spec, MFCC, acoustic)
            mel_spec, mfcc, acoustic = feature_extractor(waveform)

            feature_cache[fp] = {
                'w2v': w2v_emb,
                'mel': mel_spec,
                'mfcc': mfcc,
                'acoustic': acoustic,
                'waveform': waveform,
            }
        except Exception as e:
            dummy_mel, dummy_mfcc, dummy_acoustic = feature_extractor(np.zeros(max_samples))
            feature_cache[fp] = {
                'w2v': np.zeros(config.wav2vec2_embed_dim),
                'mel': dummy_mel,
                'mfcc': dummy_mfcc,
                'acoustic': dummy_acoustic,
                'waveform': np.zeros(max_samples),
            }

    # Backward compat for ML feature extraction
    w2v_cache = {fp: v['w2v'] for fp, v in feature_cache.items()}

    # Free GPU memory — wav2vec2 model no longer needed
    del w2v_extractor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    cache_mb = sum(
        v['w2v'].nbytes + v['mel'].nelement() * 4 + v['mfcc'].nelement() * 4 + v['acoustic'].nelement() * 4
        for v in feature_cache.values()
    ) / 1024**2
    sample_fp = list(feature_cache.keys())[0]
    print(f"\n✅ All features cached for {len(feature_cache)} files!")
    print(f"   XLS-R: ({config.wav2vec2_embed_dim},) | Mel: {list(feature_cache[sample_fp]['mel'].shape)} | MFCC: {list(feature_cache[sample_fp]['mfcc'].shape)} | Acoustic: {list(feature_cache[sample_fp]['acoustic'].shape)}")
    print(f"   Total cache size: {cache_mb:.1f} MB")
    print(f"   🚀 Training will now run ~50x faster (no librosa/parselmouth in DataLoader!)")
else:
    feature_cache = {}
    w2v_cache = {}
    print("⚠️ No audio data — skipping feature extraction.")

In [ ]:
# ============================================
# 📦 Dataset — Cached Features + Light Augmentation
# ============================================
# ALL heavy computation (librosa, parselmouth, XLS-R) is pre-computed in Cell 12.
# This Dataset only does FAST tensor lookups + lightweight augmentation.
# Result: GPU stays at 80-100% utilization instead of 0%.

class ParkinsonsAudioDataset(Dataset):
    """Voice dataset using pre-cached features. No librosa in __getitem__."""
    
    def __init__(self, dataframe: pd.DataFrame, feature_cache: Dict,
                 config: AudioConfig, augment: bool = False):
        self.df = dataframe.reset_index(drop=True)
        self.feature_cache = feature_cache
        self.config = config
        self.augment = augment
        
        # SpecAugment transforms (fast, GPU-friendly)
        if config.spec_augment and augment:
            self.freq_mask = T.FrequencyMasking(freq_mask_param=config.freq_mask_param)
            self.time_mask = T.TimeMasking(time_mask_param=config.time_mask_param)
        else:
            self.freq_mask = None
            self.time_mask = None
    
    def __len__(self):
        return len(self.df)
    
    def apply_spec_augment(self, mel_spec: torch.Tensor) -> torch.Tensor:
        if self.freq_mask is not None and self.augment:
            mel_spec = self.freq_mask(mel_spec)
            mel_spec = self.time_mask(mel_spec)
        return mel_spec
    
    def light_augment(self, mel_spec, mfcc, acoustic, w2v_emb):
        """Fast augmentations on cached tensors (no librosa needed)."""
        if not self.augment:
            return mel_spec, mfcc, acoustic, w2v_emb
        
        # SpecAugment on mel-spectrogram
        mel_spec = self.apply_spec_augment(mel_spec)
        
        # Gaussian noise on features (fast)
        if random.random() < 0.3:
            noise_scale = 0.01
            mfcc = mfcc + torch.randn_like(mfcc) * noise_scale
            acoustic = acoustic + torch.randn_like(acoustic) * noise_scale
        
        # Feature dropout (randomly zero some features)
        if random.random() < 0.2:
            mask = torch.bernoulli(torch.ones_like(mfcc) * 0.9)
            mfcc = mfcc * mask
        
        return mel_spec, mfcc, acoustic, w2v_emb
    
    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        fp = row['file_path']
        
        # Fast lookup from pre-computed cache
        cached = self.feature_cache.get(fp)
        if cached is None:
            # Fallback: zeros
            mel_spec = torch.zeros(1, self.config.n_mels, 251)
            mfcc = torch.zeros(self.config.n_mfcc * 4 if self.config.use_delta_mfcc else self.config.n_mfcc * 2)
            acoustic = torch.zeros(39)
            w2v_emb = torch.zeros(self.config.wav2vec2_embed_dim)
        else:
            mel_spec = cached['mel'].clone()
            mfcc = cached['mfcc'].clone()
            acoustic = cached['acoustic'].clone()
            w2v_emb = torch.tensor(cached['w2v'], dtype=torch.float32)
        
        # Apply fast augmentation (no disk I/O, no librosa)
        mel_spec, mfcc, acoustic, w2v_emb = self.light_augment(mel_spec, mfcc, acoustic, w2v_emb)
        
        label = torch.tensor(row['label'], dtype=torch.long)
        
        return {
            'mel_spec': mel_spec,
            'mfcc': mfcc,
            'acoustic': acoustic,
            'w2v_emb': w2v_emb,
            'label': label
        }

print("✅ Cached Dataset class defined!")
print(f"   SpecAugment: {'✅ Enabled' if config.spec_augment else '❌ Disabled'}")
print(f"   All features: ✅ From pre-computed cache (no librosa in training loop)")
print(f"   Light augment: SpecAugment + Gaussian noise + feature dropout")
print(f"   ⚡ Expected speedup: ~50x vs on-the-fly feature extraction")

In [ ]:
# ============================================
# 🏗️ Model — XLS-R + CNN + MLP Fusion with SE Attention
# ============================================

class Wav2VecAudioModel(nn.Module):
    """4-Path Fusion with Squeeze-and-Excitation (SE) attention.
    
    Architecture:
      Path 1: XLS-R embeddings (1024) → MLP → 256 dims  [multilingual, 128 languages]
      Path 2: Mel-spectrogram → CNN → 512 dims (reduced for balanced fusion)
      Path 3: MFCC+delta (160) → MLP → 128 dims
      Path 4: Acoustic features (39) → MLP → 64 dims
      SE Attention: learns to re-weight fused features by importance
      Fusion: SE(concat(256+512+128+64)) → 256 → 128 → 2
    """
    
    def __init__(self, config: AudioConfig, n_mfcc_features: int = None, n_acoustic_features: int = None):
        super().__init__()
        
        # Auto-calculate feature dims
        if n_mfcc_features is None:
            n_mfcc_features = config.n_mfcc * 2
            if config.use_delta_mfcc:
                n_mfcc_features += config.n_mfcc * 2
        
        if n_acoustic_features is None:
            n_acoustic_features = 14
            if config.use_spectral_contrast:
                n_acoustic_features += 7
            if config.use_chroma:
                n_acoustic_features += 12
            if config.use_tonnetz:
                n_acoustic_features += 6
        
        self.n_mfcc_features = n_mfcc_features
        self.n_acoustic_features = n_acoustic_features
        self.w2v_dim = config.wav2vec2_embed_dim
        
        # Path 1: XLS-R embedding MLP (1024 → 256)
        self.w2v_encoder = nn.Sequential(
            nn.Linear(config.wav2vec2_embed_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(config.dropout * 0.5)
        )
        
        # Path 2: CNN for Mel-Spectrograms
        channels = [32, 64, 128]
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, channels[0], kernel_size=3, padding=1),
            nn.BatchNorm2d(channels[0]),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels[0], channels[1], kernel_size=3, padding=1),
            nn.BatchNorm2d(channels[1]),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(channels[1], channels[2], kernel_size=3, padding=1),
            nn.BatchNorm2d(channels[2]),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((2, 2)),
            nn.Dropout2d(0.3)
        )
        cnn_output_dim = channels[2] * 2 * 2  # 512 (balanced with wav2vec2 256-dim)
        
        # Path 3: MLP for MFCC features
        self.mfcc_encoder = nn.Sequential(
            nn.Linear(n_mfcc_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(config.dropout * 0.5)
        )
        
        # Path 4: MLP for acoustic features
        self.acoustic_encoder = nn.Sequential(
            nn.Linear(n_acoustic_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )
        
        # Fusion layer: w2v(256) + cnn(512) + mfcc(128) + acoustic(64) = 960
        fusion_dim = 256 + cnn_output_dim + 128 + 64
        
        # Squeeze-and-Excitation (SE) attention block
        # Learns channel-wise importance — re-weights features globally
        se_reduction = 8
        self.se = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim // se_reduction),
            nn.ReLU(),
            nn.Linear(fusion_dim // se_reduction, fusion_dim),
            nn.Sigmoid()
        )
        
        # Classification head
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, config.hidden_dim),
            nn.BatchNorm1d(config.hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(config.dropout * 0.7),
            nn.Linear(128, 2)
        )
    
    def forward(self, mel_spec, mfcc, acoustic, w2v_emb):
        # Handle batch_size=1 during training (BatchNorm requires >1)
        single_sample = mel_spec.size(0) == 1 and self.training
        if single_sample:
            self.eval()
        
        # Path 1: Wav2Vec2
        x_w2v = self.w2v_encoder(w2v_emb)
        
        # Path 2: CNN
        x_cnn = self.conv1(mel_spec)
        x_cnn = self.conv2(x_cnn)
        x_cnn = self.conv3(x_cnn)
        x_cnn = x_cnn.view(x_cnn.size(0), -1)
        
        # Path 3: MFCC
        x_mfcc = self.mfcc_encoder(mfcc)
        
        # Path 4: Acoustic
        x_acoustic = self.acoustic_encoder(acoustic)
        
        # Fusion with SE Attention
        x_fused = torch.cat([x_w2v, x_cnn, x_mfcc, x_acoustic], dim=1)
        se_weights = self.se(x_fused)
        x_fused = x_fused * se_weights  # Channel-wise re-weighting
        
        logits = self.fusion(x_fused)
        
        if single_sample:
            self.train()
        
        return {'logits': logits}

# Test model creation
_test_model = Wav2VecAudioModel(config)
_n_params = sum(p.numel() for p in _test_model.parameters())
_fusion_dim = 256 + 512 + 128 + 64
print("✅ XLS-R + CNN + MLP Fusion Model with SE Attention defined!")
print(f"   Path 1: XLS-R embedding ({config.wav2vec2_embed_dim} → 256)")
print(f"   Path 2: Mel-spec CNN (128×T → 512)")
print(f"   Path 3: MFCC MLP ({_test_model.n_mfcc_features} → 128)")
print(f"   Path 4: Acoustic MLP ({_test_model.n_acoustic_features} → 64)")
print(f"   SE Attention: squeeze-and-excitation on {_fusion_dim}-dim fused features")
print(f"   Fusion: SE({_fusion_dim}) → {config.hidden_dim} → 128 → 2")
print(f"   Total parameters: {_n_params:,}")
del _test_model

In [ ]:
# ============================================
# CELL 12: Loss Functions (Fixed)
# ============================================

class FocalLoss(nn.Module):
    """Focal Loss with per-class alpha for class imbalance.
    
    Standard focal loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    where alpha_t = alpha for positive class (PD), (1-alpha) for negative (HC).
    
    FIX: Previously used class_weights AND alpha together (double-weighting).
    Now uses only per-class alpha_t — the correct formulation.
    """
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, labels):
        ce_loss = F.cross_entropy(logits, labels, reduction='none',
                                  label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        # Per-class alpha: alpha for PD (label=1), (1-alpha) for HC (label=0)
        alpha_t = torch.where(labels == 1, self.alpha, 1.0 - self.alpha)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

class AdvancedLoss(nn.Module):
    """Combined focal loss — no double class-weighting."""
    def __init__(self, use_focal=True, focal_alpha=0.75, focal_gamma=2.0,
                 label_smoothing=0.1):
        super().__init__()
        self.use_focal = use_focal
        if use_focal:
            self.focal = FocalLoss(alpha=focal_alpha, gamma=focal_gamma,
                                   label_smoothing=label_smoothing)
        else:
            self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, outputs, labels):
        if self.use_focal:
            loss = self.focal(outputs['logits'], labels)
        else:
            loss = self.ce(outputs['logits'], labels)
        return {'total_loss': loss, 'cls_loss': loss}

print("✅ Loss functions defined!")
print("   FocalLoss: per-class alpha (configurable, DL uses α=0.5 since sampler balances batches)")
print("   FIX: Removed double class-weighting (class_weights + alpha)")

In [ ]:
# ============================================
# 🏋️ Training & Evaluation Functions (with Mixup)
# ============================================

def mixup_data(batch_dict, labels, alpha=0.4):
    """Apply Mixup augmentation — creates virtual training examples.
    
    Mixup linearly interpolates pairs of samples and their labels,
    proven to improve generalization on small medical datasets.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
        lam = max(lam, 1 - lam)  # Ensure lam >= 0.5 for stability
    else:
        return batch_dict, labels, labels, 1.0

    batch_size = labels.size(0)
    index = torch.randperm(batch_size).to(labels.device)

    mixed = {}
    for key, val in batch_dict.items():
        mixed[key] = lam * val + (1 - lam) * val[index]

    return mixed, labels, labels[index], lam


def mixup_criterion(criterion, outputs, y_a, y_b, lam):
    """Compute loss for mixup: weighted combination of losses for both labels."""
    loss_a = criterion(outputs, y_a)['total_loss']
    loss_b = criterion(outputs, y_b)['total_loss']
    return {'total_loss': lam * loss_a + (1 - lam) * loss_b}


def train_one_epoch(model, loader, criterion, optimizer, scheduler=None,
                    use_mixup=False, mixup_alpha=0.4):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        optimizer.zero_grad()

        mel_spec = batch['mel_spec'].to(DEVICE)
        mfcc = batch['mfcc'].to(DEVICE)
        acoustic = batch['acoustic'].to(DEVICE)
        w2v_emb = batch['w2v_emb'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        if use_mixup and np.random.random() < 0.5:
            # Apply mixup 50% of the time for regularization
            batch_dict = {'mel_spec': mel_spec, 'mfcc': mfcc,
                          'acoustic': acoustic, 'w2v_emb': w2v_emb}
            mixed, y_a, y_b, lam = mixup_data(batch_dict, labels, alpha=mixup_alpha)
            outputs = model(mixed['mel_spec'], mixed['mfcc'],
                           mixed['acoustic'], mixed['w2v_emb'])
            losses = mixup_criterion(criterion, outputs, y_a, y_b, lam)
        else:
            outputs = model(mel_spec, mfcc, acoustic, w2v_emb)
            losses = criterion(outputs, labels)

        loss = losses['total_loss']

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        all_preds.extend(outputs['logits'].argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    pd_recall = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    hc_recall = recall_score(all_labels, all_preds, pos_label=0, zero_division=0)

    return total_loss / len(loader), acc, pd_recall, hc_recall

@torch.no_grad()
def evaluate_model(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    for batch in loader:
        mel_spec = batch['mel_spec'].to(DEVICE)
        mfcc = batch['mfcc'].to(DEVICE)
        acoustic = batch['acoustic'].to(DEVICE)
        w2v_emb = batch['w2v_emb'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        outputs = model(mel_spec, mfcc, acoustic, w2v_emb)
        losses = criterion(outputs, labels)

        total_loss += losses['total_loss'].item()
        probs = F.softmax(outputs['logits'], dim=-1)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

    all_preds = (np.array(all_probs) >= threshold).astype(int)
    acc = accuracy_score(all_labels, all_preds)

    pd_rec = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    hc_rec = recall_score(all_labels, all_preds, pos_label=0, zero_division=0)
    balanced_acc = (pd_rec + hc_rec) / 2.0
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    if len(set(all_labels)) > 1:
        try:
            auc = roc_auc_score(all_labels, all_probs)
        except:
            auc = 0.5
    else:
        auc = 0.5

    return {
        'loss': total_loss / max(1, len(loader)),
        'accuracy': acc,
        'balanced_accuracy': balanced_acc,
        'f1_macro': f1_macro,
        'auc': auc,
        'pd_recall': pd_rec,
        'hc_recall': hc_rec,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }

def optimize_threshold_f1(val_labels, val_probs):
    """Find optimal threshold that maximizes macro F1 score."""
    best_threshold = 0.5
    best_f1 = 0

    for threshold in np.arange(0.1, 0.9, 0.02):
        preds = (np.array(val_probs) >= threshold).astype(int)
        f1 = f1_score(val_labels, preds, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold

print("✅ Training and evaluation functions defined!")
print("   ✨ NEW: Mixup augmentation (creates virtual training samples)")
print("   Model inputs: mel_spec + mfcc + acoustic + w2v_emb")
print("   Model selection: Best macro F1 score")
print("   Threshold optimization: Maximizes macro F1")

In [ ]:
# ============================================
# 🚀 Main Training Loop — XLS-R + Enhanced ML Ensemble
# ============================================

import time

def extract_features_for_ml(df, feature_cache, config):
    """Extract XLS-R + hand-crafted features for classical ML from pre-computed cache."""
    all_mfcc, all_acoustic, all_w2v, labels = [], [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="   Extracting features", leave=False):
        fp = row['file_path']
        cached = feature_cache.get(fp)
        if cached is not None:
            all_mfcc.append(cached['mfcc'].numpy() if isinstance(cached['mfcc'], torch.Tensor) else cached['mfcc'])
            all_acoustic.append(cached['acoustic'].numpy() if isinstance(cached['acoustic'], torch.Tensor) else cached['acoustic'])
            all_w2v.append(cached['w2v'])
        else:
            mfcc_dim = config.n_mfcc * 2 + (config.n_mfcc * 2 if config.use_delta_mfcc else 0)
            ac_dim = 14 + (7 if config.use_spectral_contrast else 0) + (12 if config.use_chroma else 0) + (6 if config.use_tonnetz else 0)
            all_mfcc.append(np.zeros(mfcc_dim))
            all_acoustic.append(np.zeros(ac_dim))
            all_w2v.append(np.zeros(config.wav2vec2_embed_dim))
        labels.append(row['label'])

    X = np.hstack([np.array(all_mfcc), np.array(all_acoustic), np.array(all_w2v)])
    y = np.array(labels)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, y


def run_enhanced_cv(cv_splits, feature_extractor, w2v_cache, config):
    """Enhanced K-Fold CV: Wav2Vec2 DL + RF + SVM + XGB + GB + LGB with stacking."""

    all_preds, all_labels, all_probs = [], [], []
    all_ml_preds, all_ml_labels = [], []
    fold_results = []
    overall_start = time.time()

def run_enhanced_cv(cv_splits, feature_cache, config):
    """Enhanced K-Fold CV: XLS-R DL + RF + SVM + XGB + GB + LGB with stacking."""
    print("   DL: XLS-R_MLP + CNN + MFCC_MLP + Acoustic_MLP (language-agnostic)")
    print("   ML: RF + SVM + XGBoost + GradientBoosting + LightGBM")
    print("   Meta: Stacking ensemble (LogisticRegression)")
    print(f"   Total folds: {len(cv_splits)} | Epochs/fold: {config.num_epochs} | Patience: {config.patience}")
    print("="*70)

    for fold_idx, split in enumerate(cv_splits):
        fold_name = split['fold_name']
        fold_start = time.time()
        print(f"\n{'='*60}")
        print(f"📊 {fold_name} ({fold_idx+1}/{len(cv_splits)})")
        print(f"{'='*60}")

        train_labels = split['train_df']['label'].values
        class_counts = np.bincount(train_labels, minlength=2)
        imbalance_ratio = class_counts[0] / max(class_counts[1], 1)

        test_labels_arr = split['test_df']['label'].values
        n_test_pd = (test_labels_arr == 1).sum()
        n_test_hc = (test_labels_arr == 0).sum()
        print(f"   Train: PD={class_counts[1]}, HC={class_counts[0]} (ratio={imbalance_ratio:.1f}:1)")
        print(f"   Val: {len(split['val_df'])} samples | Test: {len(split['test_df'])} (PD={n_test_pd}, HC={n_test_hc})")

        # SKIP folds with no PD in test — can't evaluate PD detection!
        if n_test_pd == 0 or n_test_hc == 0:
            print(f"\n   ⚠️  SKIPPING {fold_name}: test set has PD={n_test_pd}, HC={n_test_hc}")
            print(f"       Need both classes in test to evaluate meaningfully.")
            continue

        # ============ PART A: Classical ML Ensemble ============
        print(f"\n   🌲 [ML] Extracting features & training classifiers...")
        ml_start = time.time()

        X_train, y_train = extract_features_for_ml(split['train_df'], feature_extractor, w2v_cache, config)
        X_val, y_val = extract_features_for_ml(split['val_df'], feature_extractor, w2v_cache, config)
        X_test, y_test = extract_features_for_ml(split['test_df'], feature_extractor, w2v_cache, config)

        print(f"      Feature shape: {X_train.shape} (train), {X_val.shape} (val), {X_test.shape} (test)")

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_train, y_train = extract_features_for_ml(split['train_df'], feature_cache, config)
        X_val, y_val = extract_features_for_ml(split['val_df'], feature_cache, config)
        X_test, y_test = extract_features_for_ml(split['test_df'], feature_cache, config)
        # SMOTE oversampling — balance PD/HC for ML training
        if SMOTE_AVAILABLE and np.sum(y_train == 1) >= 2:
            smote_k = min(3, np.sum(y_train == 1) - 1)
            smote = SMOTE(k_neighbors=smote_k, random_state=config.seed)
            X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)
            sm_ratio = np.sum(y_train_sm == 0) / max(np.sum(y_train_sm == 1), 1)
            print(f"      SMOTE: {len(y_train)} → {len(y_train_sm)} (PD: {np.sum(y_train==1)} → {np.sum(y_train_sm==1)})")
        else:
            X_train_sm, y_train_sm = X_train_s, y_train
            sm_ratio = imbalance_ratio

        # Train ML models on SMOTE-balanced features
        rf = RandomForestClassifier(
            n_estimators=500, max_depth=12, min_samples_leaf=2, max_features='sqrt',
            class_weight='balanced', random_state=config.seed, n_jobs=-1
        )
        svm = SVC(
            kernel='rbf', C=10.0, gamma='scale',
            class_weight='balanced', probability=True, random_state=config.seed
        )
        gb = GradientBoostingClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            min_samples_leaf=3, subsample=0.8, random_state=config.seed
        )
        xgb_model = xgb.XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            scale_pos_weight=sm_ratio, subsample=0.8,
            colsample_bytree=0.8, min_child_weight=3,
            eval_metric='logloss', random_state=config.seed, verbosity=0
        )

        rf.fit(X_train_sm, y_train_sm)
        svm.fit(X_train_sm, y_train_sm)
        gb.fit(X_train_sm, y_train_sm)
        xgb_model.fit(X_train_sm, y_train_sm)

        ml_models = {'RF': rf, 'SVM': svm, 'GB': gb, 'XGB': xgb_model}

        # LightGBM — often outperforms XGBoost on small medical datasets
        if LGB_AVAILABLE:
            lgb_model = lgb.LGBMClassifier(
                n_estimators=300, max_depth=5, learning_rate=0.05,
                is_unbalance=False, subsample=0.8,
                colsample_bytree=0.8, min_child_samples=3,
                random_state=config.seed, verbosity=-1
            )
            lgb_model.fit(X_train_sm, y_train_sm)
            ml_models['LGB'] = lgb_model

        val_ml_probs = {}
        test_ml_probs = {}
        for name, clf in ml_models.items():
            val_ml_probs[name] = clf.predict_proba(X_val_s)[:, 1]
            test_ml_probs[name] = clf.predict_proba(X_test_s)[:, 1]

        ml_avg_test = np.mean(list(test_ml_probs.values()), axis=0)
        ml_avg_val = np.mean(list(val_ml_probs.values()), axis=0)
        ml_threshold = optimize_threshold_f1(y_val, ml_avg_val)
        ml_preds = (ml_avg_test >= ml_threshold).astype(int)
        ml_f1 = f1_score(y_test, ml_preds, average='macro', zero_division=0)

        ml_elapsed = time.time() - ml_start
        print(f"      ML training completed in {ml_elapsed:.1f}s")
        for name, probs in test_ml_probs.items():
            th = optimize_threshold_f1(y_val, val_ml_probs[name])
            preds = (probs >= th).astype(int)
            f1_val = f1_score(y_test, preds, average='macro', zero_division=0)
            pd_r = recall_score(y_test, preds, pos_label=1, zero_division=0)
            hc_r = recall_score(y_test, preds, pos_label=0, zero_division=0)
            print(f"      {name:4s}: F1={f1_val:.3f}, PD_Rec={pd_r:.1%}, HC_Rec={hc_r:.1%}")
        print(f"      AVG:  F1={ml_f1:.3f}")

        all_ml_preds.extend(ml_preds)
        all_ml_labels.extend(y_test)

        # ============ PART B: Deep Learning (Wav2Vec2 + CNN + MLP) ============
        print(f"\n   🧠 [DL] Training XLS-R+CNN+MLP model...")

        train_dataset = ParkinsonsAudioDataset(split['train_df'], feature_cache, config, augment=True)
        val_dataset = ParkinsonsAudioDataset(split['val_df'], feature_cache, config, augment=False)
        test_dataset = ParkinsonsAudioDataset(split['test_df'], feature_cache, config, augment=False)

        train_sampler = make_weighted_sampler(train_labels)
        _pin = torch.cuda.is_available()
        train_loader = DataLoader(train_dataset, batch_size=config.batch_size, sampler=train_sampler, num_workers=2, drop_last=True, pin_memory=_pin)
        val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=_pin)
        test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=_pin)

        print(f"      Batches per epoch: {len(train_loader)} (train), {len(val_loader)} (val)")

        model = Wav2VecAudioModel(config).to(DEVICE)

        # IMPORTANT: Since WeightedRandomSampler already balances batches to ~50/50,
        # use focal_alpha=0.5 (balanced). Otherwise alpha=0.75 + sampler = double-weighting.
        effective_alpha = 0.5
        print(f"      Focal Loss: alpha={effective_alpha} (balanced — sampler handles imbalance)")
        criterion = AdvancedLoss(use_focal=True, focal_alpha=effective_alpha,
                                 focal_gamma=config.focal_gamma, label_smoothing=config.label_smoothing)
        optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=config.learning_rate,
            total_steps=len(train_loader) * config.num_epochs,
            pct_start=0.2, anneal_strategy='cos'
        )

        best_val_f1 = 0
        best_state = None
        best_val_probs_dl = None
        best_val_labels_dl = None
        patience_counter = 0

        # ── Epoch-level training log ──
        print(f"\n      {'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | {'Val F1':>6} | {'Val Acc':>7} | {'PD Rec':>6} | {'HC Rec':>6} | {'LR':>10} | {'Status'}")
        print(f"      {'-'*5}-+-{'-'*10}-+-{'-'*8}-+-{'-'*6}-+-{'-'*7}-+-{'-'*6}-+-{'-'*6}-+-{'-'*10}-+-{'-'*15}")

        dl_start = time.time()
        for epoch in range(config.num_epochs):
            epoch_start = time.time()

            train_loss, train_acc, train_pd_rec, train_hc_rec = train_one_epoch(
                model, train_loader, criterion, optimizer, scheduler,
                use_mixup=config.use_mixup, mixup_alpha=config.mixup_alpha
            )
            val_metrics = evaluate_model(model, val_loader, criterion)
            val_f1 = val_metrics['f1_macro']
            epoch_time = time.time() - epoch_start
            current_lr = optimizer.param_groups[0]['lr']

            # Determine status
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                best_val_probs_dl = val_metrics['probabilities']
                best_val_labels_dl = val_metrics['labels']
                patience_counter = 0
                status = f"⭐ Best (patience 0/{config.patience})"
            else:
                patience_counter += 1
                status = f"   (patience {patience_counter}/{config.patience})"

            print(f"      {epoch+1:>5} | {train_loss:>10.4f} | {val_metrics['loss']:>8.4f} | {val_f1:>6.3f} | {val_metrics['accuracy']:>7.3f} | {val_metrics['pd_recall']:>6.1%} | {val_metrics['hc_recall']:>6.1%} | {current_lr:>10.2e} | {status}  [{epoch_time:.1f}s]")

            if patience_counter >= config.patience:
                print(f"\n      ⏹️  Early stopping triggered at epoch {epoch+1} (no improvement for {config.patience} epochs)")
                print(f"      Best val F1: {best_val_f1:.4f}")
                break

        dl_elapsed = time.time() - dl_start
        total_epochs_run = epoch + 1
        print(f"\n      DL training done: {total_epochs_run} epochs in {dl_elapsed:.1f}s ({dl_elapsed/total_epochs_run:.1f}s/epoch)")
        print(f"      Best validation F1: {best_val_f1:.4f}")

        if best_state is not None:
            model.load_state_dict(best_state)

        dl_threshold = optimize_threshold_f1(best_val_labels_dl, best_val_probs_dl) if best_val_probs_dl else 0.5
        dl_metrics = evaluate_model(model, test_loader, criterion, threshold=dl_threshold)
        dl_probs = np.array(dl_metrics['probabilities'])
        dl_f1 = f1_score(y_test, dl_metrics['predictions'], average='macro', zero_division=0)
        print(f"      DL Test: F1={dl_f1:.3f}, PD={dl_metrics['pd_recall']:.1%}, HC={dl_metrics['hc_recall']:.1%}, Threshold={dl_threshold:.2f}")

        # ============ PART C: Stacking Meta-Learner ============
        print(f"\n   🏆 [STACK] Training stacking meta-learner...")

        dl_val_probs = np.array(best_val_probs_dl) if best_val_probs_dl else np.full(len(y_val), 0.5)

        # Dynamic stacking — adapts to available models (RF, SVM, GB, XGB, optionally LGB)
        stack_val_cols = [val_ml_probs[name] for name in ml_models]
        stack_val_cols.append(dl_val_probs)
        stack_val = np.column_stack(stack_val_cols)
        stack_test_cols = [test_ml_probs[name] for name in ml_models]
        stack_test_cols.append(dl_probs)
        stack_test = np.column_stack(stack_test_cols)

        meta = LogisticRegression(class_weight='balanced', C=1.0, random_state=config.seed, max_iter=1000)
        meta.fit(stack_val, y_val)

        ensemble_probs = meta.predict_proba(stack_test)[:, 1]
        ens_threshold = optimize_threshold_f1(y_val, meta.predict_proba(stack_val)[:, 1])
        ensemble_preds = (ensemble_probs >= ens_threshold).astype(int)

        ens_acc = accuracy_score(y_test, ensemble_preds)
        ens_f1 = f1_score(y_test, ensemble_preds, average='macro', zero_division=0)
        ens_pd = recall_score(y_test, ensemble_preds, pos_label=1, zero_division=0)
        ens_hc = recall_score(y_test, ensemble_preds, pos_label=0, zero_division=0)

        print(f"      Stacking: F1={ens_f1:.3f}, PD={ens_pd:.1%}, HC={ens_hc:.1%}")

        # Fallback: simple weighted average
        simple_ens_probs = 0.55 * ml_avg_test + 0.45 * dl_probs
        simple_th = optimize_threshold_f1(y_val, 0.55 * ml_avg_val + 0.45 * dl_val_probs)
        simple_preds = (simple_ens_probs >= simple_th).astype(int)
        simple_f1 = f1_score(y_test, simple_preds, average='macro', zero_division=0)

        if ens_f1 >= simple_f1:
            final_preds = ensemble_preds
            final_probs = ensemble_probs
            final_f1 = ens_f1
            method = "Stacking"
        else:
            final_preds = simple_preds
            final_probs = simple_ens_probs
            final_f1 = simple_f1
            method = "Weighted Avg"

        final_acc = accuracy_score(y_test, final_preds)
        final_pd = recall_score(y_test, final_preds, pos_label=1, zero_division=0)
        final_hc = recall_score(y_test, final_preds, pos_label=0, zero_division=0)

        fold_elapsed = time.time() - fold_start
        print(f"\n   ➡️  Best: {method} | F1={final_f1:.3f} | Acc={final_acc:.3f} | PD={final_pd:.1%} | HC={final_hc:.1%}")
        print(f"   ⏱️  Fold {fold_idx+1} completed in {fold_elapsed:.1f}s")

        all_preds.extend(final_preds)
        all_labels.extend(y_test)
        all_probs.extend(final_probs)

        model_path = Path(config.checkpoint_dir) / f"fold_{fold_idx+1}_model.pth"
        torch.save({
            'model_state_dict': best_state,
            'fold': fold_idx + 1,
            'fold_name': fold_name,
            'val_f1': best_val_f1,
            'test_f1': final_f1,
            'config': config
        }, model_path)
        print(f"   💾 Model checkpoint saved: {model_path}")

        fold_results.append({
            'fold_name': fold_name,
            'accuracy': final_acc,
            'f1': final_f1,
            'ml_f1': ml_f1,
            'dl_f1': dl_f1,
            'stack_f1': ens_f1,
            'pd_recall': final_pd,
            'hc_recall': final_hc,
            'n_samples': len(split['test_df']),
            'method': method,
            'model_path': str(model_path)
        })

        # ── Fold summary comparison ──
        print(f"\n   📋 Fold {fold_idx+1} Summary:")
        print(f"      {'Method':<15} | {'F1':>6} | {'PD Rec':>6} | {'HC Rec':>6}")
        print(f"      {'-'*15}-+-{'-'*6}-+-{'-'*6}-+-{'-'*6}")
        print(f"      {'ML Ensemble':<15} | {ml_f1:>6.3f} | {'':>6} | {'':>6}")
        print(f"      {'DL Model':<15} | {dl_f1:>6.3f} | {dl_metrics['pd_recall']:>6.1%} | {dl_metrics['hc_recall']:>6.1%}")
        print(f"      {'Stacking':<15} | {ens_f1:>6.3f} | {ens_pd:>6.1%} | {ens_hc:>6.1%}")
        print(f"      {'Weighted Avg':<15} | {simple_f1:>6.3f} | {'':>6} | {'':>6}")
        print(f"      {'>> Winner':<15} | {final_f1:>6.3f} | {final_pd:>6.1%} | {final_hc:>6.1%}  ({method})")

    # ── Overall summary across all folds ──
    overall_elapsed = time.time() - overall_start

    if not fold_results:
        print("\n\n❌ No valid folds were evaluated! All folds lacked PD or HC in test set.")
        print("   → Your dataset has too few PD patient groups for cross-validation.")
        print("   → Try reducing n_folds or collecting more PD patient data.")
        return [], [], [], [], [], []

    print(f"\n\n{'='*70}")
    print(f"📊 OVERALL RESULTS ACROSS {len(fold_results)} EVALUATED FOLDS")
    print(f"{'='*70}")
    print(f"   Total time: {overall_elapsed:.1f}s ({overall_elapsed/60:.1f} min)")
    print(f"\n   {'Fold':<20} | {'F1':>6} | {'ML F1':>6} | {'DL F1':>6} | {'PD Rec':>6} | {'HC Rec':>6} | {'Method'}")
    print(f"   {'-'*20}-+-{'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*12}")
    for r in fold_results:
        print(f"   {r['fold_name']:<20} | {r['f1']:>6.3f} | {r['ml_f1']:>6.3f} | {r['dl_f1']:>6.3f} | {r['pd_recall']:>6.1%} | {r['hc_recall']:>6.1%} | {r['method']}")
    avg_f1 = np.mean([r['f1'] for r in fold_results])
    avg_pd = np.mean([r['pd_recall'] for r in fold_results])
    avg_hc = np.mean([r['hc_recall'] for r in fold_results])
    print(f"   {'AVERAGE':<20} | {avg_f1:>6.3f} | {'':>6} | {'':>6} | {avg_pd:>6.1%} | {avg_hc:>6.1%} |")

    return all_preds, all_labels, all_probs, fold_results, all_ml_preds, all_ml_labels


# ============ RUN ============
if len(df_audio) > 0:
    feature_extractor = AudioFeatureExtractor(config)

    all_preds, all_labels, all_probs, fold_results, all_ml_preds, all_ml_labels = \
        run_enhanced_cv(cv_splits, feature_extractor, w2v_cache, config)
else:
    print("⚠️ No audio data available.")

In [ ]:
# ============================================
# CELL 15: Overall Results & Visualization
# ============================================

if len(df_audio) > 0 and len(all_labels) > 0 and len(fold_results) > 0:
    print("\n" + "="*70)
    print("🏆 FINAL RESULTS — Enhanced Voice-Based Parkinson's Detection")
    print("="*70)
    
    overall_acc = accuracy_score(all_labels, all_preds)
    overall_bal_acc = balanced_accuracy_score(all_labels, all_preds)
    overall_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    overall_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    overall_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    overall_f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    try:
        overall_auc = roc_auc_score(all_labels, all_probs)
    except:
        overall_auc = 0.5
    
    print(f"\n📊 Ensemble Performance (Stacking: RF+SVM+XGB+GB+DL):")
    print(f"   Accuracy:          {overall_acc:.4f} ({overall_acc*100:.2f}%)")
    print(f"   Balanced Accuracy: {overall_bal_acc:.4f} ({overall_bal_acc*100:.2f}%)")
    print(f"   Precision:         {overall_precision:.4f}")
    print(f"   Recall:            {overall_recall:.4f}")
    print(f"   F1-Score (weighted): {overall_f1:.4f}")
    print(f"   F1-Score (macro):    {overall_f1_macro:.4f}")
    print(f"   AUC-ROC:           {overall_auc:.4f}")
    
    # ML-only comparison
    ml_acc = accuracy_score(all_ml_labels, all_ml_preds)
    ml_f1_m = f1_score(all_ml_labels, all_ml_preds, average='macro', zero_division=0)
    ml_bal = balanced_accuracy_score(all_ml_labels, all_ml_preds)
    print(f"\n📊 ML Only (RF+SVM+XGB+GB average):")
    print(f"   Balanced Accuracy: {ml_bal:.4f}")
    print(f"   F1-Score (macro):  {ml_f1_m:.4f}")
    
    # Per-class
    pd_recall = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    pd_precision = precision_score(all_labels, all_preds, pos_label=1, zero_division=0)
    pd_f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
    hc_recall = recall_score(all_labels, all_preds, pos_label=0, zero_division=0)
    hc_precision = precision_score(all_labels, all_preds, pos_label=0, zero_division=0)
    hc_f1 = f1_score(all_labels, all_preds, pos_label=0, zero_division=0)
    
    print(f"\n📋 Per-Class:")
    print(f"   Parkinson's: Prec={pd_precision:.4f}, Recall={pd_recall:.4f}, F1={pd_f1:.4f}")
    print(f"   Healthy:     Prec={hc_precision:.4f}, Recall={hc_recall:.4f}, F1={hc_f1:.4f}")
    
    print(f"\n📋 Per-Fold:")
    print("-"*85)
    print(f"   {'Fold':8s} {'Best':>8s} {'ML':>8s} {'DL':>8s} {'Stack':>8s} {'PD_Rec':>8s} {'HC_Rec':>8s} {'Method':>12s}")
    print("-"*85)
    for r in fold_results:
        print(f"   {r['fold_name']:8s} {r['f1']:.4f}   {r['ml_f1']:.4f}   {r['dl_f1']:.4f}   "
              f"{r['stack_f1']:.4f}   {r['pd_recall']:.1%}    {r['hc_recall']:.1%}    {r['method']:>12s}")
    
    avg_f1 = np.mean([r['f1'] for r in fold_results])
    print(f"\n   Average F1 across folds: {avg_f1:.4f}")
    print("="*70)

In [ ]:
# ============================================
# 📊 Confusion Matrix & Classification Report
# ============================================

if len(df_audio) > 0:
    from sklearn.metrics import ConfusionMatrixDisplay
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    cm = confusion_matrix(all_labels, all_preds)
    class_names = ['Healthy', "Parkinson's"]
    
    # Raw confusion matrix
    disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp1.plot(ax=axes[0], cmap='Blues')
    axes[0].set_title('Confusion Matrix (Stratified K-Fold CV)', fontweight='bold')
    
    # Normalized
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
    disp2.plot(ax=axes[1], cmap='Blues', values_format='.2%')
    axes[1].set_title('Normalized Confusion Matrix', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('audio_lopo_confusion_matrix.png', dpi=150)
    plt.show()
    
    print("\n📋 Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0))

In [ ]:
# ============================================
# CELL 17: ROC Curve
# ============================================

if len(df_audio) > 0 and len(set(all_labels)) > 1:
    from sklearn.metrics import roc_curve
    
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC Curve (AUC = {overall_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve - Audio-Based Parkinson Detection', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('audio_roc_curve.png', dpi=150)
    plt.show()

In [ ]:
# ============================================
# 💾 Save Results & Best Model
# ============================================

if len(df_audio) > 0:
    results = {
        'method': 'XLS-R Multilingual Transfer Learning + Stacking Ensemble (RF+SVM+XGB+GB+LGB+DL)',
        'wav2vec2_model': config.wav2vec2_model_name,
        'data': f'{len(df_audio)} voice samples from Italian Parkinson\'s Voice and Speech dataset',
        'n_folds': len(fold_results),
        'ensemble_metrics': {
            'accuracy': float(overall_acc),
            'balanced_accuracy': float(overall_bal_acc),
            'f1_weighted': float(overall_f1),
            'f1_macro': float(overall_f1_macro),
            'auc_roc': float(overall_auc),
            'pd_recall': float(pd_recall),
            'hc_recall': float(hc_recall)
        },
        'ml_only': {'balanced_accuracy': float(ml_bal), 'f1_macro': float(ml_f1_m)},
        'per_fold': {r['fold_name']: {
            'f1': float(r['f1']), 'ml_f1': float(r['ml_f1']),
            'dl_f1': float(r['dl_f1']), 'method': r['method']
        } for r in fold_results}
    }
    
    with open('audio_lopo_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print("✅ Results saved to audio_lopo_results.json")
    
    best_fold = max(fold_results, key=lambda x: x['f1'])
    best_model_path = Path(config.checkpoint_dir) / "best_audio_model.pth"
    import shutil
    shutil.copy(best_fold['model_path'], best_model_path)
    
    print(f"\n" + "="*70)
    print("📊 FINAL SUMMARY — XLS-R Multilingual Transfer Learning")
    print("="*70)
    print(f"   Data:     ALL voice recordings ({len(df_audio)} samples)")
    print(f"   Method:   XLS-R + Stacking Ensemble (RF+SVM+XGB+GB+LGB+DL)")
    print(f"   Backbone: {config.wav2vec2_model_name} (128 languages)")
    print(f"   Features: XLS-R ({config.wav2vec2_embed_dim}d) + MFCC + Delta + Voice Quality + Spectral")
    print(f"   CV:       Stratified {len(fold_results)}-Fold")
    print(f"\n   ✅ Balanced Accuracy:   {overall_bal_acc*100:.2f}%")
    print(f"   ✅ PD Detection Recall: {pd_recall*100:.2f}%")
    print(f"   ✅ F1-Score (macro):    {overall_f1_macro:.4f}")
    print(f"   ✅ AUC-ROC:             {overall_auc:.4f}")
    print(f"   ✅ Best fold: {best_fold['fold_name']} (F1={best_fold['f1']:.4f})")
    print(f"\n   💾 Models: {config.checkpoint_dir}")
    print("="*70)
    
    if IN_COLAB:
        from google.colab import files
        files.download('audio_lopo_results.json')
        files.download('audio_lopo_confusion_matrix.png')
        files.download('audio_roc_curve.png')
        files.download(str(best_model_path))
else:
    print("⚠️ No results to save.")

---
## 🎯 How to Use the Saved Model for Inference

### Load Best Model:
```python
import torch
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# Load the best model
checkpoint = torch.load('checkpoints_audio/best_audio_model.pth')
model = Wav2VecAudioModel(checkpoint['config'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load XLS-R multilingual model for embedding extraction
w2v_processor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-xls-r-300m")
w2v_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m")
w2v_model.eval()

# Extract features from new audio
feature_extractor = AudioFeatureExtractor(checkpoint['config'])
waveform = librosa.load('new_audio.wav', sr=16000)[0]
mel_spec, mfcc, acoustic = feature_extractor(waveform)

# XLS-R embedding
inputs = w2v_processor(waveform, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    w2v_out = w2v_model(inputs.input_values)
    w2v_emb = w2v_out.last_hidden_state.mean(dim=1).squeeze(0)

# Predict
with torch.no_grad():
    outputs = model(mel_spec.unsqueeze(0), mfcc.unsqueeze(0), 
                    acoustic.unsqueeze(0), w2v_emb.unsqueeze(0))
    probs = F.softmax(outputs['logits'], dim=-1)
    prediction = "Parkinson's" if probs[0, 1] > 0.5 else "Healthy"
    confidence = max(probs[0, 1].item(), probs[0, 0].item())

print(f"Prediction: {prediction} (Confidence: {confidence:.2%})")
```
---